# Demo: Time-Series Modeling with statsmodels

We've got monthly electricity demand for a regional utility — 8 years of data, plus average temperature each month. The goal: forecast the next 12 months.

We'll fit two models and see how much difference it makes when you give the model information about seasonality and weather.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

HORIZON = 12

## Load data and split

Train on everything through 2023. Hold out 2024 as the test set — that's our 12-month forecast horizon.

For future temperature, we'll just use the seasonal average from training. In practice you'd plug in a weather forecast, but this is good enough for modeling.

In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date")
df = df.asfreq("MS")

train_df = df.loc[:"2023-12-01"]
test_df = df.loc["2024-01-01":]

train_demand = train_df["demand_mwh"]
train_temp = train_df["avg_temp_f"]

# Build future temperature from seasonal averages
seasonal_avg_temp = train_df.groupby(train_df.index.month)["avg_temp_f"].mean()
future_temp = pd.Series(
    [seasonal_avg_temp[m] for m in range(1, 13)],
    index=pd.date_range("2024-01-01", periods=12, freq="MS"),
)

print(f"Training: {len(train_df)} months ({train_df.index[0].date()} to {train_df.index[-1].date()})")
print(f"Test: {len(test_df)} months")

## ACF and PACF — what's the autocorrelation telling us?

Before fitting anything, let's look at the autocorrelation structure. We difference at lag 12 first to remove the annual cycle, then plot ACF and PACF on what's left.

What to look for:
- **ACF**: if there's a spike at lag 1 that cuts off, that suggests an MA(1) term
- **PACF**: spikes at lags 1-2 that cut off suggest AR(1) or AR(2)

Together, these guide the (p, d, q) order for our ARIMA model.

In [ ]:
diffed = train_demand.diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(diffed, ax=axes[0], lags=36)
plot_pacf(diffed, ax=axes[1], lags=36)
axes[0].set_title("ACF (seasonally differenced)")
axes[1].set_title("PACF (seasonally differenced)")
plt.tight_layout()
plt.show()

## OK let's fit our first model: ARIMA(2,1,1)

This is a plain ARIMA — no seasonal component, no temperature. It just looks at recent values and their differences to predict the next one.

We use `SARIMAX` from statsmodels even for ARIMA. When you leave out the seasonal order and exogenous variables, it's just ARIMA under the hood.

Pay attention to the AIC — that's our main tool for comparing models. Lower AIC = better fit (adjusted for complexity).

In [ ]:
arima_model = SARIMAX(train_demand, order=(2, 1, 1))
arima_fit = arima_model.fit(disp=False)

print(arima_fit.summary().tables[0])
print(f"\nAIC: {arima_fit.aic:.1f}")

## Now the upgrade: SARIMAX with seasonal terms + temperature

Two big improvements here:
1. **Seasonal order (1,1,1,12)** — this tells the model there's a 12-month cycle. The seasonal differencing removes the annual pattern, and seasonal AR/MA terms model what's left.
2. **Temperature as an exogenous variable** — electricity demand goes up when it's really cold (heating) and really hot (AC). Giving the model this information should tighten up the forecasts.

Watch the AIC drop.

In [ ]:
sarimax_model = SARIMAX(
    train_demand,
    exog=train_temp,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
)
sarimax_fit = sarimax_model.fit(disp=False)

print(sarimax_fit.summary().tables[0])
print(f"\nAIC: {sarimax_fit.aic:.1f}")
print(f"ARIMA AIC was: {arima_fit.aic:.1f}")
print(f"Improvement: {arima_fit.aic - sarimax_fit.aic:.1f} points")

## Forecasts with prediction intervals

`get_forecast` gives us the point forecast plus confidence intervals. The intervals are the real story here — they tell the grid operator the range of demand they need to plan for.

Let's generate both forecasts and plot them side by side.

In [ ]:
# ARIMA forecast
arima_forecast = arima_fit.get_forecast(steps=HORIZON)
arima_fc = arima_forecast.predicted_mean
arima_ci = arima_forecast.conf_int(alpha=0.05)

# SARIMAX forecast — need to pass future temperatures
sarimax_forecast = sarimax_fit.get_forecast(steps=HORIZON, exog=future_temp.values)
sarimax_fc = sarimax_forecast.predicted_mean
sarimax_ci = sarimax_forecast.conf_int(alpha=0.05)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for ax, name, fc, ci, color in [
    (axes[0], "ARIMA(2,1,1)", arima_fc, arima_ci, "tab:blue"),
    (axes[1], "SARIMAX(1,1,1)(1,1,1,12) + temperature", sarimax_fc, sarimax_ci, "tab:orange"),
]:
    ax.plot(train_demand.index, train_demand.values, color="black", label="Historical")
    ax.plot(fc.index, fc.values, color=color, linestyle="--", linewidth=2, label="Forecast")
    ax.fill_between(
        fc.index, ci.iloc[:, 0], ci.iloc[:, 1],
        color=color, alpha=0.15, label="95% interval",
    )
    ax.set_ylabel("Demand (MWh)")
    ax.set_title(name)
    ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

## What just happened

Look at the difference. The ARIMA forecast drifts flat — it has no idea about the seasonal peaks. The SARIMAX forecast actually captures the summer and winter humps, and the prediction intervals are much tighter.

Why tighter? Because temperature explains a big chunk of the variance in demand. Once the model accounts for that, there's less unexplained noise, so it's more confident in its predictions.

In the exercise, you'll fit these models yourself and dig into the diagnostics to make sure the residuals look clean.